## Config

In [1]:
import os
import csv
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from torch.amp import GradScaler, autocast

DATA_DIR = 'data'
VOCAB_PATH = os.path.join(DATA_DIR, 'vocab.txt')
META_CSV = os.path.join(DATA_DIR, 'metadata_clean.csv')
TOK_DIR = os.path.join(DATA_DIR, 'tokenized')

# ---------- Training config ----------
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SEQ_LEN = 50        # input length
STRIDE = 10         # step between windows (reduce redundancy)
BATCH_SIZE = 256
EPOCHS = 8
LR = 1e-3
GRAD_CLIP = 1.0

EMB_DIM = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 2
DROPOUT = 0.2

TRAIN_SPLIT = 0.9   # split by books
MIN_BOOK_TOKENS = 2000  # skip tiny books

STEPS_PER_EPOCH = 3000
CKPT_DIR = 'checkpoints'
SAVE_EVERY_EPOCH = True
SAVE_EVERY_STEPS = 200
# ------------------------------------

print('DEVICE:', DEVICE)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(os.path.join(CKPT_DIR, 'model2'), exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE: cuda


## Load vocab + metadata

In [2]:
def load_vocab(vocab_path: str) -> Tuple[List[str], Dict[str, int]]:
    if not os.path.exists(vocab_path):
        raise FileNotFoundError(f'vocab.txt not found: {vocab_path}')
    with open(vocab_path, 'r', encoding='utf-8') as f:
        vocab = [line.strip() for line in f if line.strip()]
    tok2id = {t: i for i, t in enumerate(vocab)}
    return vocab, tok2id

def read_metadata(meta_csv: str) -> List[dict]:
    if not os.path.exists(meta_csv):
        raise FileNotFoundError(f'metadata.csv not found: {meta_csv}')
    with open(meta_csv, 'r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))

vocab, tok2id = load_vocab(VOCAB_PATH)
rows = read_metadata(META_CSV)

pad_id = tok2id.get('<pad>', 0)
bos_id = tok2id.get('<bos>')
eos_id = tok2id.get('<eos>')

print('vocab size:', len(vocab))
print('special ids:', {'<pad>': pad_id, '<bos>': bos_id, '<eos>': eos_id})
print('metadata rows:', len(rows))

vocab size: 26286
special ids: {'<pad>': 0, '<bos>': 2, '<eos>': 3}
metadata rows: 49


## Dataset (windowed next-token prediction)

In [3]:
def load_ids_file(path: str) -> List[int]:
    with open(path, 'r', encoding='utf-8') as f:
        s = f.read().strip()
    if not s:
        return []
    return list(map(int, s.split()))

@dataclass
class BookData:
    ebook_id: str
    ids_path: str
    token_count: int

def collect_books(rows: List[dict]) -> List[BookData]:
    books = []
    for r in rows:
        ebook_id = r.get('ebook_id')
        if not ebook_id:
            continue
        ids_path = os.path.join(TOK_DIR, f'{ebook_id}.ids.txt')
        if not os.path.exists(ids_path):
            continue
        try:
            token_count = int(r.get('token_count', '0'))
        except Exception:
            token_count = 0
        if token_count < MIN_BOOK_TOKENS:
            continue
        books.append(BookData(str(ebook_id), ids_path, token_count))
    return books

def split_books(books: List[BookData], train_split: float) -> Tuple[List[BookData], List[BookData]]:
    books = books[:]
    random.shuffle(books)
    cut = int(len(books) * train_split)
    return books[:cut], books[cut:]

def load_ids_file(path: str) -> List[int]:
    with open(path, "r", encoding="utf-8") as f:
        s = f.read().strip()
    return list(map(int, s.split())) if s else []

class CachedWindowDataset(Dataset):
    def __init__(self, books, seq_len: int, stride: int):
        self.seq_len = seq_len
        self.stride = stride
        self.book_tokens: List[torch.Tensor] = []
        self.samples: List[Tuple[int, int]] = []

        for b in books:
            ids = load_ids_file(b.ids_path)
            if len(ids) < seq_len + 2:
                continue

            t = torch.tensor(ids, dtype=torch.int32)  # compact in RAM
            bi = len(self.book_tokens)
            self.book_tokens.append(t)

            max_start = t.numel() - (seq_len + 1)
            for st in range(0, max_start + 1, stride):
                self.samples.append((bi, st))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        bi, st = self.samples[idx]
        ids = self.book_tokens[bi]
        chunk = ids[st: st + self.seq_len + 1].to(torch.long)  # convert only this slice
        return chunk[:-1], chunk[1:]

books = collect_books(rows)
if not books:
    raise RuntimeError('No tokenized books found. Check data/tokenized/*.ids.txt and metadata.csv token_count.')

train_books, val_books = split_books(books, TRAIN_SPLIT)
train_ds = CachedWindowDataset(train_books, SEQ_LEN, STRIDE)
val_ds = CachedWindowDataset(val_books, SEQ_LEN, STRIDE)

print(f'Books: total={len(books)} train={len(train_books)} val={len(val_books)}')
print(f'Samples: train={len(train_ds):,} val={len(val_ds):,}')

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,     
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

Books: total=49 train=44 val=5
Samples: train=291,493 val=32,338


## Model

In [4]:
class LSTMLM(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden: int, num_layers: int, dropout: float, pad_id: int):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        e = self.emb(x)          # (B,T,E)
        out, _ = self.lstm(e)    # (B,T,H)
        out = self.drop(out)
        logits = self.fc(out)    # (B,T,V)
        return logits

In [5]:

model = LSTMLM(
    vocab_size=len(vocab),
    emb_dim=EMB_DIM,
    hidden=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_id=pad_id,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

use_amp = (DEVICE == "cuda")
scaler = GradScaler("cuda", enabled=use_amp)

sum(p.numel() for p in model.parameters())

23892142

## Sampling utilities

In [6]:
@torch.no_grad()
def sample_ids(model: nn.Module, prompt_ids: List[int], max_new_tokens: int = 120,
               temperature: float = 1.0, top_k: int = 50) -> List[int]:
    model.eval()
    x = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        logits = model(x)[:, -1, :]  # (1,V)
        logits = logits / max(temperature, 1e-6)

        if top_k and top_k > 0:
            v, ix = torch.topk(logits, k=min(top_k, logits.size(-1)))
            probs = torch.softmax(v, dim=-1)
            next_local = torch.multinomial(probs, num_samples=1)
            next_id = ix.gather(-1, next_local)
        else:
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        next_id_int = int(next_id.item())
        x = torch.cat([x, torch.tensor([[next_id_int]], device=DEVICE)], dim=1)

        if eos_id is not None and next_id_int == eos_id:
            break
        if x.size(1) > 512:
            x = x[:, -512:]

    return x[0].tolist()

def decode(ids: List[int]) -> str:
    toks = [vocab[i] if 0 <= i < len(vocab) else '<unk>' for i in ids]
    text = ' '.join(toks)
    text = (text
            .replace(' ,', ',').replace(' .', '.').replace(' !', '!').replace(' ?', '?')
            .replace(' ;', ';').replace(' :', ':')
            .replace(' )', ')').replace('( ', '(')
            .replace(' - ', '-')
           )
    return text

In [7]:

prompt = [bos_id] if bos_id is not None else []
print(decode(sample_ids(model, prompt, max_new_tokens=40, temperature=1.0, top_k=50)))

<bos> collections collections four shown burnished infatuated howell's cayuga cane terrifying hermes wasn't presiding inhaled result inhaled blade tapped buttered buttered pellings buscar skimming oozy schools tract quicker dering manitowok sensibility macvittie circuit thence weds skimming oozy milliner bewitched crafty huff


## Train + evaluate

In [8]:
def run_eval(model) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with autocast("cuda", enabled=use_amp):
                logits = model(x)
                loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            mask = (y != pad_id)
            tok = int(mask.sum().item())
            total_tokens += tok
            total_loss += float(loss.item()) * tok

    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(20.0, avg_loss))
    return avg_loss, ppl

In [9]:
global_step = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    seen_tokens = 0

    pbar = tqdm(enumerate(train_loader, start=1), total=STEPS_PER_EPOCH, desc=f"Epoch {epoch}/{EPOCHS}")
    for step, (x, y) in pbar:
        global_step += 1
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=use_amp):
            logits = model(x)
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

        scaler.scale(loss).backward()

        # if you clip grads, you must unscale first
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        scaler.step(optimizer)   # <-- this is the optimizer step
        scaler.update()

        # logging
        mask = (y != pad_id)
        tok = int(mask.sum().item())
        running += float(loss.item()) * tok
        seen_tokens += tok

        avg = running / max(1, seen_tokens)
        ppl = math.exp(min(20.0, avg))
        pbar.set_postfix(loss=f"{avg:.4f}", ppl=f"{ppl:.2f}")

        if global_step % SAVE_EVERY_STEPS == 0:
            ckpt_path = os.path.join(CKPT_DIR, f"lstm_step{global_step}.pt")
            torch.save({"model_state": model.state_dict(), "vocab": vocab}, ckpt_path)
            tqdm.write(f"[saved] {ckpt_path}")

        if step >= STEPS_PER_EPOCH:
            break

    pbar.close()
    train_loss = running / max(1, seen_tokens)
    val_loss, val_ppl = run_eval(model)
    print(f"\n== epoch {epoch} done ==")
    print(f"train_loss={train_loss:.4f} train_ppl={math.exp(min(20.0, train_loss)):.2f}")
    print(f"val_loss={val_loss:.4f}   val_ppl={val_ppl:.2f}")

    # sample (make sure sample_ids sets model.eval() inside, or do it here)
    prompt = [bos_id] if bos_id is not None else []
    out = sample_ids(model, prompt, max_new_tokens=120, temperature=1.0, top_k=50)
    print("\n[sample]")
    print(decode(out))

    if SAVE_EVERY_EPOCH:
        ckpt_path = os.path.join(CKPT_DIR, f"lstm_lm_epoch{epoch}.pt")
        torch.save({
            "model_state": model.state_dict(),
            "vocab": vocab,
            "config": {
                "SEQ_LEN": SEQ_LEN,
                "STRIDE": STRIDE,
                "EMB_DIM": EMB_DIM,
                "HIDDEN_SIZE": HIDDEN_SIZE,
                "NUM_LAYERS": NUM_LAYERS,
                "DROPOUT": DROPOUT,
            }
        }, ckpt_path)
        print(f"[saved] {ckpt_path}\n")

Epoch 1/8:   0%|          | 0/3000 [00:00<?, ?it/s]

[saved] checkpoints\lstm_step200.pt
[saved] checkpoints\lstm_step400.pt
[saved] checkpoints\lstm_step600.pt
[saved] checkpoints\lstm_step800.pt
[saved] checkpoints\lstm_step1000.pt

== epoch 1 done ==
train_loss=5.6298 train_ppl=278.62
val_loss=5.0033   val_ppl=148.90

[sample]
<bos> the <unk> of his eyes had still been given her, " the man and king, and see him, she told the people with the court of his horse; and all, the king of the way she went, and of the same time they saw his name. " the king, " said the master the queen, " i shall take the other that. the old-tale <unk>. i were like a tree. she was no more more the place of the floor and his eyes as the day and bade a golden thousand. she came out to the old bird. the sun was in a certain
[saved] checkpoints\lstm_lm_epoch1.pt



Epoch 2/8:   0%|          | 0/3000 [00:00<?, ?it/s]

[saved] checkpoints\lstm_step1200.pt
[saved] checkpoints\lstm_step1400.pt
[saved] checkpoints\lstm_step1600.pt
[saved] checkpoints\lstm_step1800.pt
[saved] checkpoints\lstm_step2000.pt
[saved] checkpoints\lstm_step2200.pt

== epoch 2 done ==
train_loss=4.9603 train_ppl=142.63
val_loss=4.7439   val_ppl=114.88

[sample]
<bos> it is more, and if there are no longer? then, but it are to the earth. it is not as i am that i see we may do? " they said, " will i never have to be sure for me; as ye call me so. " when her father had happened to him and said, " i can find that i should have done to a big one. " he asked her, but the second sister saw the old man. " i will think it, " when the next year, still he entered the ground, the very <unk>-tree, " no the
[saved] checkpoints\lstm_lm_epoch2.pt



Epoch 3/8:   0%|          | 0/3000 [00:00<?, ?it/s]

...

[saved] checkpoints\lstm_step5800.pt
[saved] checkpoints\lstm_step6000.pt
[saved] checkpoints\lstm_step6200.pt
[saved] checkpoints\lstm_step6400.pt
[saved] checkpoints\lstm_step6600.pt
[saved] checkpoints\lstm_step6800.pt

== epoch 6 done ==
train_loss=4.0960 train_ppl=60.10
val_loss=4.3089   val_ppl=74.36

[sample]
<bos> his story of gelert, of the <unk> of wales. parallels.-- the jataka and a demon of <unk>, are also derived over an egg. it seems not to the same class, or to a certain amount of evidence to make a present, which makes it much possible to give them, so do our <unk> care to be the good man. " " i am a young man, " said the woodcutter, " i will get the gold for you. " " i am told, " replied miss vernon; " there's that of my dog? " " oh, yes; my dear! you
[saved] checkpoints\lstm_lm_epoch6.pt



Epoch 7/8:   0%|          | 0/3000 [00:00<?, ?it/s]

[saved] checkpoints\lstm_step7000.pt
[saved] checkpoints\lstm_step7200.pt
[saved] checkpoints\lstm_step7400.pt
[saved] checkpoints\lstm_step7600.pt
[saved] checkpoints\lstm_step7800.pt

== epoch 7 done ==
train_loss=3.9378 train_ppl=51.31
val_loss=4.2970   val_ppl=73.48

[sample]
<bos> the popular story-book by european parallels. the tartars and european tales. the tales is a veritable acquisition. when the children of the household were to pass, one of mankind-- the other who wanted a story for it to a little. " the king gives the king an ounce of money to my daughter, " said he. " i cannot be able to die with your love, for i have not harmed you with yourself, for you have no alternative-- such as a princess, for she has left me and set me in quest of the princess. " " not, " said the
[saved] checkpoints\lstm_lm_epoch7.pt



Epoch 8/8:   0%|          | 0/3000 [00:00<?, ?it/s]

[saved] checkpoints\lstm_step8000.pt
[saved] checkpoints\lstm_step8200.pt
[saved] checkpoints\lstm_step8400.pt
[saved] checkpoints\lstm_step8600.pt
[saved] checkpoints\lstm_step8800.pt
[saved] checkpoints\lstm_step9000.pt

== epoch 8 done ==
train_loss=3.8012 train_ppl=44.75
val_loss=4.2984   val_ppl=73.58

[sample]
<bos> " the <unk> " runs " dies, " is apt to be added one! the <unk> o ' norroway i had been so long, and could get only the last, but i knew the tooting way that the old woman would never hear. ' ' ah, ' said the old fellow, ' but he's quite a fool but this is the story. now, if i don't lose your last you will take it, ' the other admitted him. " after supper i have gone up with me, and there you may think he would ask some night to eat some sort of thing ever-- the bear
[saved] checkpoints\lstm_lm_epoch8.pt



## Tests

In [10]:
def encode_prompt(text: str) -> List[int]:
    toks = text.split()
    ids = [tok2id.get(t, tok2id.get('<unk>', 0)) for t in toks]
    return ids

In [11]:
prompt = "the older fairies stood all in a group , saying loudly"
prompt_ids = encode_prompt(prompt)
out = sample_ids(model, prompt_ids, temperature=0.9, top_k=50)
print(decode(out))

the older fairies stood all in a group, saying loudly as the old man drew out her nose-up. " oh, i believe, " said he to himself; and then he said, " no, no! i am not sure i can. well! " " yes, " said he, " i'm going to stop with me. " the poor boy did not know what to do. so he went to the king and said: " the story of which i will now find it. " " let me come, " answered matthias, as he drew him into the tower and said: " dear one, i will give you a present -


In [12]:
prompt = "once upon a time"
prompt_ids = encode_prompt(prompt)
out = sample_ids(model, prompt_ids, temperature=0.9, top_k=50)
print(decode(out))

once upon a time, at first, edward had only entered the cottage, and at last he found himself a long afternoon, and there he was lying in the middle of the loch. at last frank looked back with his hands, and looked up at him with this voice, and spoke upon his friend, ' the children of lir are too heavy to hear what did they. it had been long and still more. they were indeed the king, nor had done that they had been so powerful and powerful, and was too much to do with them in all the land. they found their bodies were filled with rage. and


## Model 2

In [13]:
model2 = LSTMLM(
    vocab_size=len(vocab),
    emb_dim=EMB_DIM,
    hidden=768,
    num_layers=NUM_LAYERS,
    dropout=0.3,
    pad_id=pad_id,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = torch.optim.Adam(model2.parameters(), lr=LR, weight_decay=1e-4)

use_amp = (DEVICE == "cuda")
scaler = GradScaler("cuda", enabled=use_amp)

sum(p.numel() for p in model2.parameters())

34819758

In [14]:
prompt = [bos_id] if bos_id is not None else []
print(decode(sample_ids(model2, prompt, max_new_tokens=40, temperature=1.0, top_k=50)))

<bos> disasters marshalled northward sentimental wailer deepened entreated shirts oxford offense kulhwych antics notorious empanelled leelinau corresponding conduct tasty erring richard's rabbits proclaiming comet alive ain petals darlings fictions unhallowed handkerchiefs engrossed entreated rapidity sweetmeat notorious spendthrift wont daren't proclaiming shuddering


In [15]:
global_step = 0
MODEL2_EPOCHS = 12
MODEL2_STEPS_PER_EPOCH = 5000

for epoch in range(1, MODEL2_EPOCHS + 1):
    model2.train()
    running = 0.0
    seen_tokens = 0

    pbar = tqdm(enumerate(train_loader, start=1), total=MODEL2_STEPS_PER_EPOCH, desc=f"Epoch {epoch}/{MODEL2_EPOCHS}")
    for step, (x, y) in pbar:
        global_step += 1
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=use_amp):
            logits = model2(x)
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

        scaler.scale(loss).backward()

        # if you clip grads, you must unscale first
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model2.parameters(), GRAD_CLIP)

        scaler.step(optimizer)   # <-- this is the optimizer step
        scaler.update()

        # logging
        mask = (y != pad_id)
        tok = int(mask.sum().item())
        running += float(loss.item()) * tok
        seen_tokens += tok

        avg = running / max(1, seen_tokens)
        ppl = math.exp(min(20.0, avg))
        pbar.set_postfix(loss=f"{avg:.4f}", ppl=f"{ppl:.2f}")

        if global_step % 500 == 0:
            ckpt_path = os.path.join(CKPT_DIR, f"model2/lstm_step{global_step}.pt")
            torch.save({"model_state": model2.state_dict(), "vocab": vocab}, ckpt_path)
            tqdm.write(f"[saved] {ckpt_path}")

        if step >= MODEL2_STEPS_PER_EPOCH:
            break

    pbar.close()
    train_loss = running / max(1, seen_tokens)
    val_loss, val_ppl = run_eval(model2)
    print(f"\n== epoch {epoch} done ==")
    print(f"train_loss={train_loss:.4f} train_ppl={math.exp(min(20.0, train_loss)):.2f}")
    print(f"val_loss={val_loss:.4f}   val_ppl={val_ppl:.2f}")

    # sample (make sure sample_ids sets model.eval() inside, or do it here)
    prompt = [bos_id] if bos_id is not None else []
    out = sample_ids(model2, prompt, max_new_tokens=120, temperature=1.0, top_k=50)
    print("\n[sample]")
    print(decode(out))

    if SAVE_EVERY_EPOCH:
        ckpt_path = os.path.join(CKPT_DIR, f"model2/lstm_lm_epoch{epoch}.pt")
        torch.save({
            "model_state": model2.state_dict(),
            "vocab": vocab,
            "config": {
                "SEQ_LEN": SEQ_LEN,
                "STRIDE": STRIDE,
                "EMB_DIM": EMB_DIM,
                "HIDDEN_SIZE": 768,
                "NUM_LAYERS": NUM_LAYERS,
                "DROPOUT": 0.3,
            }
        }, ckpt_path)
        print(f"[saved] {ckpt_path}\n")

Epoch 1/12:   0%|          | 0/5000 [00:00<?, ?it/s]

[saved] checkpoints\model2/lstm_step500.pt
[saved] checkpoints\model2/lstm_step1000.pt

== epoch 1 done ==
train_loss=5.4880 train_ppl=241.77
val_loss=4.8547   val_ppl=128.34

[sample]
<bos>. " no, i do not know, " replied the giant. " i can you want to live it, and i shall tell me to the king, " but said, but all the other would look and said. so they called the ground the prince of the first, and they was now by the same time, and then began to go up his hands. he reached her head, but all the king was very sad and even-- his sister. he was glad the greatest, who has got back the ground and had brought his sword; but this time he had only done for the
[saved] checkpoints\model2/lstm_lm_epoch1.pt



Epoch 2/12:   0%|          | 0/5000 [00:00<?, ?it/s]

[saved] checkpoints\model2/lstm_step1500.pt
[saved] checkpoints\model2/lstm_step2000.pt

== epoch 2 done ==
train_loss=4.8391 train_ppl=126.36
val_loss=4.6072   val_ppl=100.20

[sample]
<bos> the <unk>-bearer. the <unk> and his soul grew at its eyes. yet after the time had been killed, the gods, who thought there came a boy, and the two three days, a week that looked at the village, but no one could carry a whole day for all the people that were in with her heart. they would not get, and when when they saw the princess sent to his palace, he gave the queen to her with a great lady. " but a long i am glad to the world, that it will not only for the way. " " but let me go
[saved] checkpoints\model2/lstm_lm_epoch2.pt



Epoch 3/12:   0%|          | 0/5000 [00:00<?, ?it/s]

...

[saved] checkpoints\model2/lstm_step10500.pt
[saved] checkpoints\model2/lstm_step11000.pt

== epoch 10 done ==
train_loss=4.4934 train_ppl=89.42
val_loss=4.4147   val_ppl=82.66

[sample]
<bos>! " she felt. so i had already put me. " then he was, and he left-- by one of his two eyes, of the wild little bird-- one who had been seen in the world. as in the same, an indian story, which the story has taken (<unk>, or the most kind <unk> of the most of <unk>., as on the first one in which the most popular were the stories of the story of this story is of the popular version of the <unk>-book. the second form is full the original the story from the story by the most of
[saved] checkpoints\model2/lstm_lm_epoch10.pt



Epoch 11/12:   0%|          | 0/5000 [00:00<?, ?it/s]

[saved] checkpoints\model2/lstm_step11500.pt
[saved] checkpoints\model2/lstm_step12000.pt
[saved] checkpoints\model2/lstm_step12500.pt

== epoch 11 done ==
train_loss=4.4954 train_ppl=89.61
val_loss=4.4035   val_ppl=81.73

[sample]
<bos> the poor woman was about her way when they saw her poor beauty, that he gave up the money to drink, and carried it by bread to come before her. he had been afraid to tell him what the good was had told him. when he came to a town that was found he was a very pretty boy. he started and down it, which was all that seemed to save him. this was the way to see what else this was to be done. they all made about their poor fellow, and the mother gave him to some great milk that he could never see one day. at the
[saved] checkpoints\model2/lstm_lm_epoch11.pt



Epoch 12/12:   0%|          | 0/5000 [00:00<?, ?it/s]

[saved] checkpoints\model2/lstm_step13000.pt
[saved] checkpoints\model2/lstm_step13500.pt

== epoch 12 done ==
train_loss=4.4966 train_ppl=89.71
val_loss=4.4075   val_ppl=82.06

[sample]
<bos> the house. " " the two people were well! " they replied. " and we shall catch, " said the prince, who he had sent a ship for his hand, but on the way he had no great fear for he could save him into the world. now, without a loss of such trouble. he had not to leave him in this palace, that he had received a little white beard in the air, or before the prince was able to look for him, who was an old man, and in another way. and in this moment, they heard the voice of the man,
[saved] checkpoints\model2/lstm_lm_epoch12.pt



## Tests 2

In [16]:
prompt = "once upon a time"
prompt_ids = encode_prompt(prompt)
out = sample_ids(model2, prompt_ids, temperature=0.9, top_k=50)
print(decode(out))

once upon a time there, and by the next day of his wife, " the princess, " the tsar said, " i'll go out of your bed and see the other. now, what may i do? " and the old man gave the way by. " if we are going about, " he said, " i've now here all the men in a hut. if you should i need be at the end of the king's house, " " i've got a good time to-morrow, " said the young man. " this is the best thing it could. my brother, and he can have a little


## Hyperparameter Grid Search

In [17]:
import itertools

# ---------- Search space ----------
SEARCH_GRID = {
    'lr':          [5e-4, 1e-3, 2e-3],
    'hidden_size': [256, 512, 768],
    'dropout':     [0.2, 0.3, 0.5],
}

SEARCH_EMB_DIM    = 256
SEARCH_NUM_LAYERS = 2
SEARCH_EPOCHS     = 8
SEARCH_STEPS_PER_EPOCH = 1500
SEARCH_BATCH_SIZE = 256
SEARCH_SEQ_LEN    = 50
SEARCH_STRIDE     = 10

# Build all configs
keys = list(SEARCH_GRID.keys())
combos = list(itertools.product(*[SEARCH_GRID[k] for k in keys]))
configs = [dict(zip(keys, c)) for c in combos]

print(f"Total configurations: {len(configs)}")
for i, cfg in enumerate(configs):
    print(f"  [{i+1}] lr={cfg['lr']}, hidden={cfg['hidden_size']}, dropout={cfg['dropout']}")

Total configurations: 27
  [1] lr=0.0005, hidden=256, dropout=0.2
  [2] lr=0.0005, hidden=256, dropout=0.3
  [3] lr=0.0005, hidden=256, dropout=0.5
  [4] lr=0.0005, hidden=512, dropout=0.2
  [5] lr=0.0005, hidden=512, dropout=0.3
  [6] lr=0.0005, hidden=512, dropout=0.5
  [7] lr=0.0005, hidden=768, dropout=0.2
  [8] lr=0.0005, hidden=768, dropout=0.3
  [9] lr=0.0005, hidden=768, dropout=0.5
  [10] lr=0.001, hidden=256, dropout=0.2
  [11] lr=0.001, hidden=256, dropout=0.3
  [12] lr=0.001, hidden=256, dropout=0.5
  [13] lr=0.001, hidden=512, dropout=0.2
  [14] lr=0.001, hidden=512, dropout=0.3
  [15] lr=0.001, hidden=512, dropout=0.5
  [16] lr=0.001, hidden=768, dropout=0.2
  [17] lr=0.001, hidden=768, dropout=0.3
  [18] lr=0.001, hidden=768, dropout=0.5
  [19] lr=0.002, hidden=256, dropout=0.2
  [20] lr=0.002, hidden=256, dropout=0.3
  [21] lr=0.002, hidden=256, dropout=0.5
  [22] lr=0.002, hidden=512, dropout=0.2
  [23] lr=0.002, hidden=512, dropout=0.3
  [24] lr=0.002, hidden=512, dro

In [18]:
def run_search_eval(mdl) -> Tuple[float, float]:
    """Evaluate model on val set, return (avg_loss, perplexity)."""
    mdl.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            with autocast("cuda", enabled=use_amp):
                logits = mdl(x)
                loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            mask = (y != pad_id)
            tok = int(mask.sum().item())
            total_tokens += tok
            total_loss += float(loss.item()) * tok
    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(20.0, avg_loss))
    return avg_loss, ppl


search_results = []

for cfg_idx, cfg in enumerate(configs):
    print(f"\n{'='*60}")
    print(f"Config [{cfg_idx+1}/{len(configs)}]  lr={cfg['lr']}  hidden={cfg['hidden_size']}  dropout={cfg['dropout']}")
    print('='*60)

    set_seed(SEED)

    mdl = LSTMLM(
        vocab_size=len(vocab),
        emb_dim=SEARCH_EMB_DIM,
        hidden=cfg['hidden_size'],
        num_layers=SEARCH_NUM_LAYERS,
        dropout=cfg['dropout'],
        pad_id=pad_id,
    ).to(DEVICE)

    crit = nn.CrossEntropyLoss(ignore_index=pad_id)
    opt  = torch.optim.Adam(mdl.parameters(), lr=cfg['lr'])
    scl  = GradScaler("cuda", enabled=use_amp)

    best_val_loss = float('inf')

    for epoch in range(1, SEARCH_EPOCHS + 1):
        mdl.train()
        running = 0.0
        seen_tokens = 0

        pbar = tqdm(enumerate(train_loader, start=1),
                     total=SEARCH_STEPS_PER_EPOCH,
                     desc=f"  E{epoch}/{SEARCH_EPOCHS}", leave=False)

        for step, (x, y) in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=use_amp):
                logits = mdl(x)
                loss = crit(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            scl.scale(loss).backward()
            scl.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), GRAD_CLIP)
            scl.step(opt)
            scl.update()

            mask = (y != pad_id)
            tok = int(mask.sum().item())
            running += float(loss.item()) * tok
            seen_tokens += tok

            avg = running / max(1, seen_tokens)
            pbar.set_postfix(loss=f"{avg:.4f}")

            if step >= SEARCH_STEPS_PER_EPOCH:
                break
        pbar.close()

        val_loss, val_ppl = run_search_eval(mdl)
        best_val_loss = min(best_val_loss, val_loss)
        print(f"  epoch {epoch}  train_loss={running/max(1,seen_tokens):.4f}  val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}")

    search_results.append({
        **cfg,
        'best_val_loss': best_val_loss,
        'best_val_ppl': math.exp(min(20.0, best_val_loss)),
    })

    # free GPU memory before next config
    del mdl, opt, scl, crit
    torch.cuda.empty_cache()

print("\n\nGrid search complete.")


Config [1/27]  lr=0.0005  hidden=256  dropout=0.2


  E1/8:   0%|          | 0/1500 [00:00<?, ?it/s]

  epoch 1  train_loss=6.2336  val_loss=5.7521  val_ppl=314.84


  E2/8:   0%|          | 0/1500 [00:00<?, ?it/s]

  epoch 2  train_loss=5.6709  val_loss=5.3861  val_ppl=218.36


...

  epoch 6  train_loss=3.7987  val_loss=4.2710  val_ppl=71.59


  E7/8:   0%|          | 0/1500 [00:00<?, ?it/s]

  epoch 7  train_loss=3.7096  val_loss=4.3009  val_ppl=73.76


  E8/8:   0%|          | 0/1500 [00:00<?, ?it/s]

  epoch 8  train_loss=3.6377  val_loss=4.3327  val_ppl=76.15


Grid search complete.


In [19]:
# Sort by best validation loss and display results
search_results.sort(key=lambda r: r['best_val_loss'])

print(f"{'Rank':<5} {'LR':<10} {'Hidden':<8} {'Dropout':<9} {'Val Loss':<10} {'Val PPL':<10}")
print('-' * 55)
for i, r in enumerate(search_results):
    print(f"{i+1:<5} {r['lr']:<10} {r['hidden_size']:<8} {r['dropout']:<9.1f} {r['best_val_loss']:<10.4f} {r['best_val_ppl']:<10.2f}")

print(f"\nBest config:")
best = search_results[0]
print(f"  lr={best['lr']}, hidden_size={best['hidden_size']}, dropout={best['dropout']}")
print(f"  val_loss={best['best_val_loss']:.4f}, val_ppl={best['best_val_ppl']:.2f}")

Rank  LR         Hidden   Dropout   Val Loss   Val PPL   
-------------------------------------------------------
1     0.002      768      0.5       4.2496     70.08     
2     0.001      768      0.3       4.2628     71.01     
3     0.001      768      0.5       4.2657     71.21     
4     0.001      768      0.2       4.2726     71.71     
5     0.002      768      0.3       4.2770     72.02     
6     0.0005     768      0.3       4.2785     72.14     
7     0.0005     768      0.2       4.2803     72.26     
8     0.001      512      0.2       4.2848     72.59     
9     0.002      768      0.2       4.2871     72.76     
10    0.002      512      0.3       4.2884     72.85     
11    0.0005     768      0.5       4.2905     73.00     
12    0.001      512      0.3       4.2934     73.21     
13    0.002      512      0.2       4.3001     73.71     
14    0.002      512      0.5       4.3034     73.95     
15    0.001      512      0.5       4.3040     73.99     
16    0.0005    

In [20]:
import itertools

# ---------- Search space ----------
SEARCH_GRID = {
    'lr':          [0.001, 0.0015, 0.002],
    'hidden_size': [768, 1024],
    'dropout':     [0.3, 0.4, 0.5],
}

SEARCH_EMB_DIM    = 256
SEARCH_NUM_LAYERS = 2
SEARCH_EPOCHS     = 8
SEARCH_STEPS_PER_EPOCH = 1500
SEARCH_BATCH_SIZE = 256
SEARCH_SEQ_LEN    = 50
SEARCH_STRIDE     = 10

# Build all configs
keys = list(SEARCH_GRID.keys())
combos = list(itertools.product(*[SEARCH_GRID[k] for k in keys]))
configs = [dict(zip(keys, c)) for c in combos]

print(f"Total configurations: {len(configs)}")
for i, cfg in enumerate(configs):
    print(f"  [{i+1}] lr={cfg['lr']}, hidden={cfg['hidden_size']}, dropout={cfg['dropout']}")

Total configurations: 18
  [1] lr=0.001, hidden=768, dropout=0.3
  [2] lr=0.001, hidden=768, dropout=0.4
  [3] lr=0.001, hidden=768, dropout=0.5
  [4] lr=0.001, hidden=1024, dropout=0.3
  [5] lr=0.001, hidden=1024, dropout=0.4
  [6] lr=0.001, hidden=1024, dropout=0.5
  [7] lr=0.0015, hidden=768, dropout=0.3
  [8] lr=0.0015, hidden=768, dropout=0.4
  [9] lr=0.0015, hidden=768, dropout=0.5
  [10] lr=0.0015, hidden=1024, dropout=0.3
  [11] lr=0.0015, hidden=1024, dropout=0.4
  [12] lr=0.0015, hidden=1024, dropout=0.5
  [13] lr=0.002, hidden=768, dropout=0.3
  [14] lr=0.002, hidden=768, dropout=0.4
  [15] lr=0.002, hidden=768, dropout=0.5
  [16] lr=0.002, hidden=1024, dropout=0.3
  [17] lr=0.002, hidden=1024, dropout=0.4
  [18] lr=0.002, hidden=1024, dropout=0.5


In [21]:
def run_search_eval(mdl) -> Tuple[float, float]:
    """Evaluate model on val set, return (avg_loss, perplexity)."""
    mdl.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            with autocast("cuda", enabled=use_amp):
                logits = mdl(x)
                loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            mask = (y != pad_id)
            tok = int(mask.sum().item())
            total_tokens += tok
            total_loss += float(loss.item()) * tok
    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(20.0, avg_loss))
    return avg_loss, ppl


search_results = []

for cfg_idx, cfg in enumerate(configs):
    print(f"\n{'='*60}")
    print(f"Config [{cfg_idx+1}/{len(configs)}]  lr={cfg['lr']}  hidden={cfg['hidden_size']}  dropout={cfg['dropout']}")
    print('='*60)

    set_seed(SEED)

    mdl = LSTMLM(
        vocab_size=len(vocab),
        emb_dim=SEARCH_EMB_DIM,
        hidden=cfg['hidden_size'],
        num_layers=SEARCH_NUM_LAYERS,
        dropout=cfg['dropout'],
        pad_id=pad_id,
    ).to(DEVICE)

    crit = nn.CrossEntropyLoss(ignore_index=pad_id)
    opt  = torch.optim.Adam(mdl.parameters(), lr=cfg['lr'])
    scl  = GradScaler("cuda", enabled=use_amp)

    best_val_loss = float('inf')

    for epoch in range(1, SEARCH_EPOCHS + 1):
        mdl.train()
        running = 0.0
        seen_tokens = 0

        pbar = tqdm(enumerate(train_loader, start=1),
                     total=SEARCH_STEPS_PER_EPOCH,
                     desc=f"  E{epoch}/{SEARCH_EPOCHS}", leave=False)

        for step, (x, y) in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=use_amp):
                logits = mdl(x)
                loss = crit(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            scl.scale(loss).backward()
            scl.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), GRAD_CLIP)
            scl.step(opt)
            scl.update()

            mask = (y != pad_id)
            tok = int(mask.sum().item())
            running += float(loss.item()) * tok
            seen_tokens += tok

            avg = running / max(1, seen_tokens)
            pbar.set_postfix(loss=f"{avg:.4f}")

            if step >= SEARCH_STEPS_PER_EPOCH:
                break
        pbar.close()

        val_loss, val_ppl = run_search_eval(mdl)
        best_val_loss = min(best_val_loss, val_loss)
        print(f"  epoch {epoch}  train_loss={running/max(1,seen_tokens):.4f}  val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}")

    search_results.append({
        **cfg,
        'best_val_loss': best_val_loss,
        'best_val_ppl': math.exp(min(20.0, best_val_loss)),
    })

    # free GPU memory before next config
    del mdl, opt, scl, crit
    torch.cuda.empty_cache()

print("\n\nGrid search complete.")


Config [1/18]  lr=0.001  hidden=768  dropout=0.3


  E1/8:   0%|          | 0/1500 [00:00<?, ?it/s]

  epoch 1  train_loss=5.5215  val_loss=4.8654  val_ppl=129.72


  E2/8:   0%|          | 0/1500 [00:00<?, ?it/s]

  epoch 2  train_loss=4.7829  val_loss=4.5563  val_ppl=95.23


...

  epoch 6  train_loss=3.5626  val_loss=4.2940  val_ppl=73.26


  E7/8:   0%|          | 0/1500 [00:00<?, ?it/s]

  epoch 7  train_loss=3.4670  val_loss=4.3459  val_ppl=77.16


  E8/8:   0%|          | 0/1500 [00:00<?, ?it/s]

  epoch 8  train_loss=3.3904  val_loss=4.3809  val_ppl=79.91


Grid search complete.


In [22]:
# Sort by best validation loss and display results
search_results.sort(key=lambda r: r['best_val_loss'])

print(f"{'Rank':<5} {'LR':<10} {'Hidden':<8} {'Dropout':<9} {'Val Loss':<10} {'Val PPL':<10}")
print('-' * 55)
for i, r in enumerate(search_results):
    print(f"{i+1:<5} {r['lr']:<10} {r['hidden_size']:<8} {r['dropout']:<9.1f} {r['best_val_loss']:<10.4f} {r['best_val_ppl']:<10.2f}")

print(f"\nBest config:")
best = search_results[0]
print(f"  lr={best['lr']}, hidden_size={best['hidden_size']}, dropout={best['dropout']}")
print(f"  val_loss={best['best_val_loss']:.4f}, val_ppl={best['best_val_ppl']:.2f}")

Rank  LR         Hidden   Dropout   Val Loss   Val PPL   
-------------------------------------------------------
1     0.002      1024     0.5       4.2245     68.34     
2     0.0015     1024     0.5       4.2323     68.87     
3     0.002      1024     0.4       4.2330     68.92     
4     0.0015     1024     0.4       4.2381     69.28     
5     0.001      1024     0.3       4.2425     69.58     
6     0.001      1024     0.5       4.2439     69.68     
7     0.0015     1024     0.3       4.2451     69.76     
8     0.001      1024     0.4       4.2470     69.90     
9     0.002      768      0.5       4.2496     70.08     
10    0.0015     768      0.5       4.2525     70.28     
11    0.002      1024     0.3       4.2540     70.39     
12    0.0015     768      0.4       4.2582     70.68     
13    0.0015     768      0.3       4.2605     70.85     
14    0.002      768      0.4       4.2607     70.86     
15    0.001      768      0.4       4.2619     70.95     
16    0.001     